# SmartAd Performance Analysis — Machine Learning A/B Testing

---

## 📋 Overview

This notebook takes a **Machine Learning approach** to A/B analysis, going beyond
the binary yes/no question of hypothesis testing to ask:

> *Which contextual factors (device, browser, time, OS) drive brand-awareness response?*

### Why Machine Learning for A/B Testing?

Classical and Sequential tests answer **"Did the ad work?"** (causal inference).  
Machine learning answers **"Who does the ad work for, and when?"** (predictive modelling).

| Question | Method |
|---|---|
| Did the ad increase awareness? | Classical Z-test / SPRT |
| Which user attributes predict awareness? | Machine Learning |
| How large is the effect for iOS users? | ML segmentation |
| What is the most important predictive factor? | Feature importance |

### Models Used

| Model | Strength | Interpretability |
|---|---|---|
| Logistic Regression | Fast, probabilistic, feature p-values | High |
| Decision Tree | Interpretable splits, visual | Medium |
| Random Forest | Robust, handles non-linearity | Medium (importance scores) |
| XGBoost | High accuracy, handles imbalance | Medium (SHAP compatible) |

---

## 🔬 Problem Formulation

**Binary classification task**:

- **Target variable**: `aware` (1 = brand aware, 0 = not aware)
- **Features**: experiment group, hour, day, device brand, OS, browser
- **Evaluation**: Stratified 5-fold cross-validation + held-out test set
- **Metrics**: Accuracy, F1 Score, ROC-AUC

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import os
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
import mlflow.sklearn

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report, roc_curve,
)

warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'script'))

from train_classifiers import TrainingClassifier
from data_pipeline import clean, load_raw

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.family': 'DejaVu Sans', 'figure.dpi': 110})
print('✅ Imports successful')

---
## 📂 1. Load and Prepare Data

In [ ]:
# Load and clean raw data
df_raw   = load_raw('../data/AdSmart_AB_testing_data.csv')
df_clean = clean(df_raw)

print(f'\nClean dataset shape: {df_clean.shape}')
print(f'Awareness rate     : {df_clean["aware"].mean():.2%}')
print(f'Class balance      :')
print(df_clean['aware'].value_counts(normalize=True).rename({0: 'Not Aware', 1: 'Aware'}).to_string())

In [ ]:
# ── Feature engineering ───────────────────────────────────────────────────────
# Select features available after cleaning
cat_cols = [c for c in ['device_brand', 'os_name', 'browser_group'] if c in df_clean.columns]
num_cols = [c for c in ['is_exposed', 'hour', 'day_of_week'] if c in df_clean.columns]

df_ml = df_clean[num_cols + cat_cols + ['aware']].dropna()

# One-hot encode categoricals (drop first to avoid multicollinearity)
X = pd.concat([
    df_ml[num_cols].reset_index(drop=True),
    pd.get_dummies(df_ml[cat_cols], drop_first=True).reset_index(drop=True)
], axis=1)
y = df_ml['aware'].reset_index(drop=True)

print(f'Feature matrix shape : {X.shape}')
print(f'Features             : {list(X.columns[:10])} ...')

# Train-test split (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)
print(f'\nTrain: {X_train.shape}, Test: {X_test.shape}')
print(f'Train awareness rate: {y_train.mean():.2%}')
print(f'Test awareness rate : {y_test.mean():.2%}')

---
## 🤖 2. Train and Evaluate Logistic Regression

We start with Logistic Regression because it is the most interpretable:
- Coefficients tell us the direction and magnitude of each feature's effect
- We can compute feature **p-values** (statistical significance of each predictor)

In [ ]:
# Train Logistic Regression with MLflow tracking
mlflow.set_experiment('SmartAd_ML_AB_Testing')

with mlflow.start_run(run_name='logistic_regression'):
    lr_trainer = TrainingClassifier(X_train, X_test, y_train, y_test, 'logistic_regression')
    train_res  = lr_trainer.train(folds=5, track_mlflow=True)
    test_res   = lr_trainer.test(track_mlflow=True)

In [ ]:
# Confusion matrix
lr_trainer.plot_confusion_matrix(track_mlflow=False)

In [ ]:
# Feature p-values — which contextual features significantly predict awareness?
p_df = lr_trainer.get_p_values()
print('\nTop 10 most significant features:')
print(p_df.head(10).to_string(index=False))

# Visualise
lr_trainer.plot_p_values(p_df)

---
## 🌲 3. Train Random Forest

Random Forest gives us **feature importance scores** — how much each feature
contributes to reducing uncertainty in predictions.

In [ ]:
with mlflow.start_run(run_name='random_forest'):
    rf_trainer = TrainingClassifier(X_train, X_test, y_train, y_test, 'random_forest')
    train_res_rf = rf_trainer.train(folds=5, track_mlflow=True)
    test_res_rf  = rf_trainer.test(track_mlflow=True)

In [ ]:
# Feature importances
fi_df = rf_trainer.get_feature_importance()
print('\nTop 15 feature importances (Random Forest):')
print(fi_df.head(15).to_string(index=False))

rf_trainer.plot_feature_importance(top_n=15)

---
## ⚡ 4. Train XGBoost

XGBoost often achieves the highest accuracy on structured tabular data.
It uses gradient boosting to iteratively correct prediction errors.

In [ ]:
try:
    with mlflow.start_run(run_name='xgboost'):
        xgb_trainer = TrainingClassifier(X_train, X_test, y_train, y_test, 'xgboost')
        train_res_xgb = xgb_trainer.train(folds=5, track_mlflow=True)
        test_res_xgb  = xgb_trainer.test(track_mlflow=True)
        xgb_trainer.plot_feature_importance(top_n=15)
except Exception as e:
    print(f'XGBoost not available: {e}')

---
## 📊 5. Model Comparison

Compare all trained models side-by-side on the test set.

In [ ]:
# Collect results for comparison
results_table = pd.DataFrame([
    {
        'Model': 'Logistic Regression',
        'CV Accuracy': np.mean(train_res['fold_accuracies']),
        'Test Accuracy': test_res['accuracy'],
        'Test F1': test_res['f1'],
        'Test AUC': test_res['auc'],
    },
    {
        'Model': 'Random Forest',
        'CV Accuracy': np.mean(train_res_rf['fold_accuracies']),
        'Test Accuracy': test_res_rf['accuracy'],
        'Test F1': test_res_rf['f1'],
        'Test AUC': test_res_rf['auc'],
    },
])

display_cols = ['Model', 'CV Accuracy', 'Test Accuracy', 'Test F1', 'Test AUC']
print(results_table[display_cols].to_string(index=False, float_format='{:.4f}'.format))

# Bar chart comparison
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
metrics = ['Test Accuracy', 'Test F1', 'Test AUC']
colors = ['#4361ee', '#f72585', '#2a9d8f']

for ax, metric, color in zip(axes, metrics, colors):
    bars = ax.bar(results_table['Model'], results_table[metric], color=color, alpha=0.85, width=0.5)
    ax.set_title(metric, fontsize=12)
    ax.set_ylim(0, 1)
    ax.tick_params(axis='x', rotation=15)
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.02, f'{h:.3f}', ha='center', va='bottom', fontsize=10)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('Model Comparison — Test Set Metrics', fontsize=14)
plt.tight_layout()
plt.show()

---
## 🏁 6. Conclusions & Business Insights

### What the ML Models Tell Us

1. **Experiment group (`is_exposed`)** should be a significant predictor if the ad works.
   Its significance in the logistic regression p-values confirms the classical test findings.

2. **Hour of day** often emerges as an important feature — users at certain times
   may be more receptive to brand messaging.

3. **Browser type** captures media context (Facebook in-app browser vs native Chrome)
   which reflects the content environment around the ad.

4. **Device brand** correlates with user demographics (iPhone users vs Generic Android
   budget users), which affects awareness propensity.

### Recommended Actions

| Finding | Recommendation |
|---|---|
| Peak awareness hours identified | Schedule ad delivery during high-responsiveness hours |
| Certain browsers show higher awareness | Prioritise ad spend on high-converting browser environments |
| Device brand affects response | Tailor creative format to top-performing device categories |
| Exposure increases awareness | Continue the SmartAd creative — the test confirms its effectiveness |

### Limitations of ML A/B Analysis

- ML models are **correlational**, not causal — a feature being predictive doesn't mean it *causes* awareness
- Class imbalance (few `aware=1` cases) may bias model towards the majority class
- Non-respondents (yes=0, no=0) introduce **selection bias** into the training data

### Next Steps

1. Use **SHAP values** for model-agnostic feature attribution
2. Implement a **two-stage hurdle model** for non-respondent bias correction
3. Explore **Uplift Modelling** to directly model the causal treatment effect per segment